<a href="https://colab.research.google.com/github/DhikshaSubash/market-sentiment-analysis/blob/main/notebooks/03_spark_etl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 - Install dependencies
!pip install pyspark nltk -q

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

print("✅ Libraries ready!")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


✅ Libraries ready!


In [ ]:
# Cell 2 - Load raw data from Drive
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MarketSentimentETL") \
    .getOrCreate()

# Load saved parquet files
news_df = spark.read.parquet("/content/drive/MyDrive/market_sentiment/raw/news")
price_df = spark.read.parquet("/content/drive/MyDrive/market_sentiment/raw/prices")

print(f"✅ News records loaded: {news_df.count()}")
print(f"✅ Price records loaded: {price_df.count()}")
news_df.show(3, truncate=50)

Mounted at /content/drive
✅ News records loaded: 50
✅ Price records loaded: 1950
+------+--------------------------------------------------+-------------------+--------------------+--------------------------+
|symbol|                                          headline|             source|        published_at|               ingested_at|
+------+--------------------------------------------------+-------------------+--------------------+--------------------------+
|  AAPL|          openclaw-trading-assistant added to PyPI|           Pypi.org|2026-03-25T03:18:37Z|2026-03-26T11:24:31.031994|
|  AAPL|Apple’s services flywheel spins at full speed, ...|   Macdailynews.com|2026-03-24T20:30:52Z|2026-03-26T11:24:31.032011|
|  AAPL|Morgan Stanley Is Doubling Down on Apple Stock ...|Yahoo Entertainment|2026-03-24T13:49:09Z|2026-03-26T11:24:31.032015|
+------+--------------------------------------------------+-------------------+--------------------+--------------------------+
only showing top 3 rows

In [ ]:
# Cell 3 - Clean raw headlines
from pyspark.sql.functions import (
    lower, regexp_replace, trim, col
)

def clean_headlines(df):
    return df \
        .withColumn("cleaned", lower(col("headline"))) \
        .withColumn("cleaned", regexp_replace(col("cleaned"), r"http\S+", "")) \
        .withColumn("cleaned", regexp_replace(col("cleaned"), r"[^a-zA-Z\s]", "")) \
        .withColumn("cleaned", regexp_replace(col("cleaned"), r"\s+", " ")) \
        .withColumn("cleaned", trim(col("cleaned")))

news_clean_df = clean_headlines(news_df)

print("✅ Headlines cleaned!")
print("\nBefore vs After:")
news_clean_df.select("headline", "cleaned").show(3, truncate=60)

✅ Headlines cleaned!

Before vs After:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                    headline|                                                     cleaned|
+------------------------------------------------------------+------------------------------------------------------------+
|                    openclaw-trading-assistant added to PyPI|                      openclawtradingassistant added to pypi|
|Apple’s services flywheel spins at full speed, says Everc...|apples services flywheel spins at full speed says evercor...|
|Morgan Stanley Is Doubling Down on Apple Stock Thanks to ...|morgan stanley is doubling down on apple stock thanks to ...|
+------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 rows


In [ ]:
# Cell 4 - Tokenization (split headline into words)
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="cleaned", outputCol="tokens")
news_tokenized_df = tokenizer.transform(news_clean_df)

print("✅ Tokenization done!")
print("\nSample tokens:")
news_tokenized_df.select("cleaned", "tokens").show(3, truncate=60)

✅ Tokenization done!

Sample tokens:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                     cleaned|                                                      tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|                      openclawtradingassistant added to pypi|                 [openclawtradingassistant, added, to, pypi]|
|apples services flywheel spins at full speed says evercor...|[apples, services, flywheel, spins, at, full, speed, says...|
|morgan stanley is doubling down on apple stock thanks to ...|[morgan, stanley, is, doubling, down, on, apple, stock, t...|
+------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 rows


In [ ]:
# Cell 5 - Remove stop words (the, is, at, which, etc.)
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens")
news_filtered_df = remover.transform(news_tokenized_df)

print("✅ Stop words removed!")
print("\nTokens vs Filtered:")
news_filtered_df.select("tokens", "filtered_tokens").show(3, truncate=60)

✅ Stop words removed!

Tokens vs Filtered:
+------------------------------------------------------------+------------------------------------------------------------+
|                                                      tokens|                                             filtered_tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|                 [openclawtradingassistant, added, to, pypi]|                     [openclawtradingassistant, added, pypi]|
|[apples, services, flywheel, spins, at, full, speed, says...|[apples, services, flywheel, spins, full, speed, says, ev...|
|[morgan, stanley, is, doubling, down, on, apple, stock, t...|[morgan, stanley, doubling, apple, stock, thanks, stillhi...|
+------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 rows


In [ ]:
# Cell 6 - Lemmatization (running → run, stocks → stock)
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    if tokens is None:
        return []
    return [lemmatizer.lemmatize(word) for word in tokens]

lemmatize_udf = udf(lemmatize_words, ArrayType(StringType()))

news_lemmatized_df = news_filtered_df.withColumn(
    "lemmatized_tokens",
    lemmatize_udf(col("filtered_tokens"))
)

print("✅ Lemmatization done!")
print("\nFiltered vs Lemmatized:")
news_lemmatized_df.select("filtered_tokens", "lemmatized_tokens").show(3, truncate=60)

✅ Lemmatization done!

Filtered vs Lemmatized:
+------------------------------------------------------------+------------------------------------------------------------+
|                                             filtered_tokens|                                           lemmatized_tokens|
+------------------------------------------------------------+------------------------------------------------------------+
|                     [openclawtradingassistant, added, pypi]|                     [openclawtradingassistant, added, pypi]|
|[apples, services, flywheel, spins, full, speed, says, ev...|[apple, service, flywheel, spin, full, speed, say, everco...|
|[morgan, stanley, doubling, apple, stock, thanks, stillhi...|[morgan, stanley, doubling, apple, stock, thanks, stillhi...|
+------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 rows


In [ ]:
# Cell 7 - Standardize price data
from pyspark.sql.functions import round as spark_round, to_timestamp

price_clean_df = price_df \
    .withColumn("price", spark_round(col("price"), 2)) \
    .withColumn("timestamp", to_timestamp(col("timestamp"))) \
    .dropDuplicates(["symbol", "timestamp"]) \
    .orderBy("symbol", "timestamp")

print("✅ Price data cleaned!")
print(f"Price records after dedup: {price_clean_df.count()}")
price_clean_df.show(3)

✅ Price data cleaned!
Price records after dedup: 1950
+------+------+------+-------------------+
|symbol| price|volume|          timestamp|
+------+------+------+-------------------+
|  AAPL|254.28|948674|2026-03-25 13:30:00|
|  AAPL| 254.5|103492|2026-03-25 13:31:00|
|  AAPL|254.87|111227|2026-03-25 13:32:00|
+------+------+------+-------------------+
only showing top 3 rows


In [ ]:
# Cell 8 - Save processed data
# Select final columns for news
news_final_df = news_lemmatized_df.select(
    "symbol",
    "headline",
    "cleaned",
    "lemmatized_tokens",
    "source",
    "published_at",
    "ingested_at"
)

# Save both
news_final_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/processed/news"
)
price_clean_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/processed/prices"
)

print("✅ Processed data saved!")
print(f"\nFinal news records: {news_final_df.count()}")
print(f"Final price records: {price_clean_df.count()}")
print("\nFinal news schema:")
news_final_df.printSchema()

✅ Processed data saved!

Final news records: 50
Final price records: 1950

Final news schema:
root
 |-- symbol: string (nullable = true)
 |-- headline: string (nullable = true)
 |-- cleaned: string (nullable = true)
 |-- lemmatized_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- source: string (nullable = true)
 |-- published_at: string (nullable = true)
 |-- ingested_at: string (nullable = true)

